In [8]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Adding to path: {project_root}")
sys.path.append(project_root)


Adding to path: /Users/arjein/Documents/GitHub/microsoft_hackathon


In [9]:
from objects.gmail_handler import GmailHandler


gmail_handler = GmailHandler()

Authenticated as: mertarcan8@gmail.com
User: Mert Arcan
Label already exists: CLARA - IGNORED
Label already exists: CLARA - FYI
Label already exists: CLARA - NEEDS YOUR INPUT
Label already exists: CLARA - READY TO SEND


In [10]:
gmail_handler.labels_dict

{'CHAT': 'CHAT',
 'SENT': 'SENT',
 'INBOX': 'INBOX',
 'IMPORTANT': 'IMPORTANT',
 'TRASH': 'TRASH',
 'DRAFT': 'DRAFT',
 'SPAM': 'SPAM',
 'CATEGORY_FORUMS': 'CATEGORY_FORUMS',
 'CATEGORY_UPDATES': 'CATEGORY_UPDATES',
 'CATEGORY_PERSONAL': 'CATEGORY_PERSONAL',
 'CATEGORY_PROMOTIONS': 'CATEGORY_PROMOTIONS',
 'CATEGORY_SOCIAL': 'CATEGORY_SOCIAL',
 'STARRED': 'STARRED',
 'UNREAD': 'UNREAD',
 'CLARA - IGNORED': 'Label_24',
 'CLARA - FYI': 'Label_25',
 'CLARA - NEEDS YOUR INPUT': 'Label_26',
 'CLARA - READY TO SEND': 'Label_27'}

In [ ]:
label_api_response = gmail_handler.service.users().labels().list(userId='me').execute()
label_names = [label['name'] for label in label_api_response['labels']]
label_ids = [label['id'] for label in label_api_response['labels']]

In [ ]:
label_modifications = {
                'addLabelIds': 'Label_4',
            }

gmail_handler.service.users().threads().modify(id='196509cd185c95e6', userId='me', body=label_modifications).execute()


In [ ]:
all_threads = gmail_handler.fetch_all_threads()

In [ ]:

# Print current directory to understand where we are
print(f"Current directory: {os.getcwd()}")

# Add the project root to the path


# Now try the import
from secretary_agent import SecretaryAgent

In [ ]:
SecAgentInstance = SecretaryAgent()
secretary_agent = SecAgentInstance.secretary_agent

In [ ]:
all_threads[0].subject

In [ ]:
# Show the agent
from IPython.display import Image, display
display(Image(secretary_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
import os
thread_dir = os.path.join('../', "threads")
thread_files = [f for f in os.listdir(thread_dir) if f.startswith("thread_") and f.endswith(".json")]
print(f"Available thread files: {thread_files}")


In [ ]:
import json
from objects.mail_thread import MailThread

thread_file_path = os.path.join('../threads', 'thread_196509cd185c95e6.json')
with open(thread_file_path, 'r') as file:
    thread_data = json.load(file)
thread_test = MailThread.fromJson(thread_data)
thread_test.subject

In [ ]:
body = thread_test.create_prompt_for_response()

email_input = {
    'subject': thread_test.subject,
    'email_thread': body,
}


In [ ]:
response = secretary_agent.invoke({"email_input": email_input})

In [ ]:
response

In [ ]:
for m in response["messages"]:
    m.pretty_print()

In [ ]:
response['final_response']


In [ ]:
for x in response['messages'][1]:
    print(x)

In [ ]:
from email_response import EmailResponse
resp = response['final_response']
response_clean = EmailResponse.format_email(resp)


In [ ]:
print(response_clean)

In [ ]:
import base64
from email.message import EmailMessage

# Make sure thread_data has all the necessary information
print(f"Thread ID: {thread_data['id']}")
print(f"Last message ID: {thread_data['messages'][-1]['message_id']}")
print(f"Subject: {thread_data['subject']}")

# Create the draft
draft = EmailMessage()
draft.set_content(response_clean)

# Make sure the subject matches EXACTLY - Gmail is strict about this
# If the original doesn't have "Re:", don't add it
original_subject = thread_data['subject']
draft['Subject'] = original_subject

# Set proper headers for a reply
draft['In-Reply-To'] = thread_data['messages'][-1]['message_id']
# References should include the whole chain of message IDs
references = []
for message in thread_data['messages']:
    if 'message_id' in message and message['message_id']:
        references.append(message['message_id'])
draft['References'] = ' '.join(references)

# Set sender and recipient
draft['To'] = thread_data['messages'][-1]['sender_email']
draft['From'] = thread_data['messages'][-1]['recipient_email']

# Encode the message
encoded_message = base64.urlsafe_b64encode(draft.as_bytes()).decode()

# Create the message with threadId
create_message = {
    "message": {
        "raw": encoded_message,
        "threadId": thread_data['id']  # This should be in the message metadata
    },
    "threadId": thread_data['id']  # This should be in the outer request
}

# Create the draft with explicit debugging
try:
    draft = (
        service.users()
        .drafts()
        .create(userId="me", body=create_message)
        .execute()
    )
    print(f"Draft created successfully with ID: {draft.get('id')}")
    print(f"Draft thread ID: {draft.get('message', {}).get('threadId')}")
except Exception as e:
    print(f"Error creating draft: {e}")